In [1]:
from string import Template

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import minimize
from scipy.special import logsumexp, rel_entr
from scipy.stats import entropy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import MultinomialNB

In [2]:
SYSTEM_PROMPT = (
    "You are a creative-writing assistant. "
    "Write only the requested story text. Never add commentary."
)

PROMPT = Template(
    "Write a complete$genre short story of no more than 1,000 words. "
    "The story must have a clear ending. "
    "Do not include a title, preface, notes, or commentary.\n\nStory:\n"
)

GENRES = {
    "Horror": "horror",
    "Detective fiction": "detective fiction",
    "Romance": "romance",
    "Science fiction": "science fiction",
    "Thriller": "thriller",
}

genre = GENRES["Horror"]
prompt = PROMPT.substitute(genre=f" {genre}" if genre else "")
print(prompt)



Write a complete horror short story of no more than 1,000 words. The story must have a clear ending. Do not include a title, preface, notes, or commentary.

Story:



In [3]:
genre = None
prompt = PROMPT.substitute(genre=f" {genre}" if genre else "")
print(prompt)

Write a complete short story of no more than 1,000 words. The story must have a clear ending. Do not include a title, preface, notes, or commentary.

Story:



In [7]:
# load data set
df = pd.read_parquet("data/w2-d1_genre-stories.parquet")


In [8]:
df # tabluar data: ~2D matrix (rows, cols) --> spreadsheet

,genre,text
0,Horror,The first sound I heard was the scratching. It...
1,Horror,"The first time I saw the shadow, it was behind..."
2,Horror,"The power flickered, plunging the little house..."
3,Horror,The last thing I heard was the clock. Its hand...
4,Horror,The old house on Maple Lane had always stood s...
...,...,...
5995,NULL,The first thing Maya noticed was the color of ...
5996,NULL,The old house on Maple Lane was famous for thr...
5997,NULL,The old house on Maple Street had always been ...
5998,NULL,"The sky above the town of Larkspur was a pale,..."


In [9]:
print(df["text"].sample().item())

Detective Sergeant Paul Darnell watched the rain slide down the grimy window of his office at the 12th Precinct. It had been a slow week, but he was never idle. Cases came and went, but the city itself never stopped whispering.

His desk was cluttered with files and empty coffee cups. A half-smoked cigarette, the brand he’d been smoking since his last promotion, was balanced on the ashtray. He glanced at the clock. Another late night.

A knock at the door startled him. Officer Lila Torres stood there, her face pale in the flickering light. “Paul, I need your help. Someone’s been breaking into Mrs. Pendergast’s apartment on 5th and 8th. She called it in just a few minutes ago.”

Darnell lit the cigarette and took a drag. “I’ll be right there.”

The walk up the stairs to Mrs. Pendergast’s unit was a long one. The building was old, the corridors narrow, the air thick with the scent of wet wool and mildew. Mrs. Pendergast herself answered the door, her hair in curlers, her eyes wide.

“Luc

In [10]:
df["genre"].value_counts()

genre
Horror               1000
Detective fiction    1000
Romance              1000
Science fiction      1000
Thriller             1000
NULL                 1000
Name: count, dtype: int64

In [11]:
df.columns

Index(['genre', 'text'], dtype='str')

In [12]:
df["text"].sample().item()

'The first time I saw the red eyes, I thought it was the moon. It was late—too late for the neighbors, but not late enough for me, sitting on the back porch with a cup of cold coffee. The air was heavy with the scent of honeysuckle and something metallic, like old blood. The porch light was on, but I had the shades drawn, and the world was a patchwork of darkness and the distant hum of a lawnmower.\n\nI heard footsteps behind me.\n\nI spun around, heart thudding. Nothing. Just the creak of the wooden steps and the muffled sound of my own breath.\n\nThe footsteps came again, closer. I grabbed the coffee cup and pressed my back against the wall. The glass on the porch swing shivered as if someone had touched it. The shadows moved, unevenly, as if something was lurking in the darkness between the boards.\n\nI whispered, “Who’s there?”\n\nA voice, low and rough, answered. “Me.”\n\nI froze. The voice came from the empty porch. I blinked, and it was still there, staring at me from the far en

In [15]:
print(df["text"].sample().item())

The city lights flickered through the cracked window, painting the cramped apartment in shifting neon. Outside, the rain fell in sheets, muffling the distant sirens and the thrum of hovercars. Inside, Lena pressed her palms to her temples, trying to slow the pounding in her head.

She was late again. Another report, another meeting, another email that slipped through her fingers like a fish. The world spun on, relentless and uncaring.

The door chimed. Lena tensed, expecting her neighbor’s voice to echo in the hallway. Instead, it was the sound of heavy footsteps, boots on wet tile. She glanced at the clock—2:47 AM. Lena’s heart raced. She should have been asleep. She should have been anywhere but here.

The footsteps paused. Lena’s breath hitched.

“Lena,” a voice called softly, “are you awake?”

She forced a smile, blinking rapidly to clear her vision. “Yes,” she replied, her voice barely above a whisper.

The footsteps resumed, moving closer. Lena’s eyes widened as she recognized th

In [24]:
# document -term matrix: 2D matrix of term counts over documents.
#
# \ d/t \ t1 \ t2 \ t3 \
# \ d1 \ 0 \ 1 \ 0 > G2
# \ d2 \ 3 \ 0 \ 1 \ > G1*
# \ d3 \ 3 \ 0 \ 1 \ > G1*

# scikit-learn's CountVectorizer produces a document-term matrix
vectorizer = CountVectorizer(
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    stop_words="english", # common words like 'the', 'a', 'and', but also 'time', 'he', 'she'"
    max_df = 0.9, # Max doc frequency: if a word appears in 90% or more of outputs: ignore
    min_df=10, # Min document frequency: if a word appears in 10 or less outputs: ignore.
)


# scikit-learn's object: fit, transform (use what you learned to make a data representation), predict (use a fitted object to make prediction)
#
#
# model.fit()
# model.fit_predict()

X = vectorizer.fit_transform(df["text"].tolist())

feature_names = vectorizer.get_feature_names_out()
y = df["genre"].to_numpy()



In [22]:
X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1666917 stored elements and shape (6000, 8314)>

In [25]:
y

array(['Horror', 'Horror', 'Horror', ..., 'NULL', 'NULL', 'NULL'],
      shape=(6000,), dtype=object)

In [26]:
pd.DataFrame(X.toarray(), columns=feature_names) # Document-term matrix

,abandon,abandoned,abated,ability,ablaze,able,aboard,abort,abruptly,absence,...,york,young,younger,youngest,youth,zero,zipped,zipper,zone,zoomed
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5996,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
5997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
df.shape[0] == y.shape[0]

True

In [33]:
# What is the probability of token t appearing in genre g?
#
# fit a classifier that tries to estimate G_{0-5} from T_{i}, where i = document of token counts
#
# Learn a probability distribution of token counts over genres
# Then, for a given document, sum the log probability distribution of its token counts and estimate its probability vis-a-vis all genres
# Assign a genre label to the genre with the highest probability (argmax)
#
# Multinomial: ~proportional data over categories
# - Text daata  loves this kind of thing because tokens are categories
# - We used multinomial sampling over model vocabulary
# - Certain genres will share words: detective fiction and thriller --> "gun" --> what is the p(g) in DF and T?
# --> what is the p(gunn) in DF and T? What is the p(genre_DF or genre_T) given p(gun) + p (other tokens)?

clf = MultinomialNB()
clf = fit(X , y)


NameError: name 'fit' is not defined

In [32]:
np.exp(clf.class_log_prior_) # convert log probs to probs with e ** x

NameError: name 'clf' is not defined

In [35]:
# given our first document's term counts, what's the probability distribution over genre?
# genre is also multinomial, i.e., a set of categories

pd.Dataframe(clf.predict_proba{X[0]}), columns=clf.classes_) # underscores are another annoying convention scikit-learn.

SyntaxError: unmatched ')' (3974293212.py, line 1)